# Day 4: Multilingual Text Analysis

We compare two methods for classifying climate headlines in English, German,
and Spanish: separate language dictionaries and one English dictionary
applied to translated texts. The 36 headlines and translations were
constructed for teaching. They are not published news or model output.

Run every cell from top to bottom. The main path needs only the core course
packages and the bundled CSV. No API key or download is needed.


## 1. Prepared setup

Keep the original text, language, translation and human label together.


In [ ]:
from pathlib import Path
import copy
import re
import pandas as pd

def find_data_dir():
    for path in [Path("materials/data"), Path("../data"), Path("data")]:
        if (path / "multilingual_climate_examples.csv").exists():
            return path
    raise FileNotFoundError("Start JupyterLab in the course root or materials/notebooks.")

DATA_DIR = find_data_dir()
texts = pd.read_csv(DATA_DIR / "multilingual_climate_examples.csv")
assert texts["id"].is_unique
assert set(texts["lang"]) == {"en", "de", "es"}
texts.head()


In [ ]:
pd.crosstab(texts["lang"], texts["gold"])


## 2. A pooled word count

Revisit the Day 1 document-term matrix. These sentences concern a similar
policy, but their word forms do not overlap.


In [ ]:
examples = ["A new law cuts emissions",
            "Ein neues Gesetz senkt Emissionen",
            "Una nueva ley reduce las emisiones"]
for sentence in examples:
    print(sentence, ":", re.findall(r"[^\W\d_]+", sentence.casefold()))


**Discuss:** Which features would have to be connected to make these texts comparable?


## 3. Route A: dictionaries for each language

The category names are shared; each language has its own patterns. As in
Day 2, a dictionary is a transparent baseline. We leave a no-match text
or a tie uncoded.


In [ ]:
def tokenize(text):
    # Unicode letters, with punctuation and numbers separated.
    return re.findall(r"[^\W\d_]+", str(text).casefold(), flags=re.UNICODE)

# Small baseline dictionaries. The omissions give us errors to inspect.
LEXICONS = {
    "en": {
        "policy": [r"tax\w*", r"rule\w*", r"subsid\w*", r"ban\w*", r"law\w*"],
        "protest": [r"activist\w*", r"march\w*", r"rally\w*", r"protest\w*",
                    r"block\w*", r"demonstrat\w*"],
        "impact": [r"flood\w*", r"drought\w*", r"heatwave\w*", r"wildfire\w*", r"fire\w*"],
    },
    "de": {
        "policy": [r"steuer\w*", r"gesetz\w*", r"regel\w*", r"zusch\w*", r"verbot\w*"],
        "protest": [r"aktivist\w*", r"demonstr\w*", r"kundgebung\w*",
                    r"protest\w*", r"blockier\w*"],
        "impact": [r"überschwemm\w*", r"dürre\w*", r"hitzewell\w*",
                   r"waldbränd\w*", r"trockenheit\w*"],
    },
    "es": {
        "policy": [r"impuesto\w*", r"ley\w*", r"norma\w*", r"subsidio\w*", r"prohibic\w*"],
        "protest": [r"activista\w*", r"march\w*", r"manifest\w*",
                    r"protest\w*", r"bloque\w*"],
        "impact": [r"inund\w*", r"sequía\w*", r"calor\w*", r"incendio\w*", r"ola\w*"],
    },
}

def dictionary_scores(text, language, lexicons=LEXICONS):
    tokens = tokenize(text)
    return {
        label: sum(any(re.fullmatch(pattern, token) for pattern in patterns)
                   for token in tokens)
        for label, patterns in lexicons[language].items()
    }

def dictionary_label(text, language, lexicons=LEXICONS):
    scores = dictionary_scores(text, language, lexicons)
    top = max(scores.values())
    winners = [label for label, value in scores.items() if value == top]
    return winners[0] if top > 0 and len(winners) == 1 else "uncoded"

def predict_original(frame, lexicons=LEXICONS):
    return [dictionary_label(text, lang, lexicons)
            for text, lang in zip(frame["text"], frame["lang"])]

def predict_translated(frame, english=None):
    lexicons = {"en": english if english is not None else LEXICONS["en"]}
    return [dictionary_label(text, "en", lexicons) for text in frame["translation_en"]]


In [ ]:
example = "Students protest the carbon tax"
print(tokenize(example))
print(dictionary_scores(example, "en"))
print(dictionary_label(example, "en"))


In [ ]:
original_prediction = predict_original(texts)
original_view = texts[["id", "lang", "text", "gold"]].copy()
original_view["predicted"] = original_prediction
original_view


**Discuss:** Why does the German pattern for regel miss Emissionsregeln
in de02? Which errors need a new pattern, and which need a clearer codebook?


## 4. Route B: translate, then analyse

Reuse the English dictionary on supplied English paraphrases. In a real
study, save the translation system and version, then inspect translations
against their originals.


In [ ]:
texts.loc[texts["id"].isin(["de02", "de11", "es11"]),
          ["id", "text", "translation_en", "gold"]]


In [ ]:
translated_prediction = predict_translated(texts)
translated_view = texts[["id", "lang", "translation_en", "gold"]].copy()
translated_view["predicted"] = translated_prediction
translated_view


**Discuss:** The original de11 has Hitzewellen, while its English
paraphrase has extreme heat. The first dictionary finds an impact;
the English heatwave pattern misses it.


## 5. Compare against human labels

Coverage is the share receiving a category. Accuracy counts uncoded as an
error. With four texts per category and language, these scores demonstrate
an evaluation workflow, not performance on a real corpus.


In [ ]:
def compare_routes(frame, original, translated):
    left = frame.assign(method="Original dictionaries", predicted=original)
    right = frame.assign(method="English translations", predicted=translated)
    results = pd.concat([left, right], ignore_index=True)
    results["correct"] = results["predicted"].eq(results["gold"])
    results["coded"] = results["predicted"].ne("uncoded")
    return results

def language_summary(results):
    return (results.groupby(["method", "lang"], sort=False)
            .agg(n=("id", "size"), coverage=("coded", "mean"),
                 accuracy=("correct", "mean")).reset_index())


In [ ]:
results = compare_routes(texts, original_prediction, translated_prediction)
language_summary(results)


In [ ]:
categories = ["policy", "protest", "impact", "uncoded"]
for (method, lang), group in results.groupby(["method", "lang"], sort=False):
    print("\n", method, "|", lang)
    print(pd.crosstab(
        pd.Categorical(group["gold"], categories=categories),
        pd.Categorical(group["predicted"], categories=categories),
        rownames=["Human"], colnames=["Model"], dropna=False))


In [ ]:
errors = results.loc[~results["correct"],
                     ["method", "id", "lang", "text", "translation_en", "gold", "predicted"]]
errors.sort_values(["lang", "id", "method"])


In [ ]:
paired = results.pivot(index="id", columns="method", values="predicted")
paired = texts[["id", "lang", "gold", "text"]].merge(
    paired, left_on="id", right_index=True)
paired.loc[paired["Original dictionaries"] != paired["English translations"]]


**Interpret:** Which language-by-category errors would change a cross-country comparison? Report the affected document IDs and per-language results, not only pooled accuracy.


## 6. Optional extension: multilingual sentence embeddings

Slides 47–54 use a measured example from `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. The English emissions-law query had cosine similarity **0.954** to a German near-translation, **0.890** to a Spanish near-translation, and **0.247** to an unrelated German tax-protest text. Those values demonstrate cross-language retrieval; they are not policy labels. A second example gave **0.819** similarity between English carbon-tax sentences making opposite claims about families. Topic similarity is not stance agreement.

The cell below reproduces both examples and then finds cross-language neighbours in the constructed course corpus. It is skipped in a normal Run All. To run it live, install `sentence-transformers` and download the named model before class. The required dictionary walkthrough above needs neither package nor download.


In [ ]:
RUN_EMBEDDINGS = False
if RUN_EMBEDDINGS:
    import numpy as np
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

    # Reproduce the cross-language retrieval example on slide 57.
    demo = [
        "Parliament passes a new law to reduce greenhouse-gas emissions.",
        "Das Parlament verabschiedet ein neues Gesetz zur Senkung der Treibhausgasemissionen.",
        "El parlamento aprueba una ley para reducir las emisiones de gases de efecto invernadero.",
        "Studierende protestieren gegen eine neue Steuer.",
    ]
    demo_vectors = model.encode(demo, normalize_embeddings=True)
    display(pd.DataFrame({
        "candidate": demo[1:],
        "cosine_to_English_query": (demo_vectors[1:] @ demo_vectors[0]).round(3),
    }))

    # Similarity can be high even when the claim has the opposite direction.
    stance = [
        "The carbon tax will protect low-income families.",
        "The carbon tax will hurt low-income families.",
        "Die CO2-Steuer wird Familien mit geringem Einkommen schaden.",
        "A football team won the championship.",
    ]
    stance_vectors = model.encode(stance, normalize_embeddings=True)
    display(pd.DataFrame({
        "candidate": stance[1:],
        "cosine_to_protect_claim": (stance_vectors[1:] @ stance_vectors[0]).round(3),
    }))

    # Apply the same idea to the constructed course corpus.
    vectors = model.encode(texts["text"].tolist(), normalize_embeddings=True)
    similarities = vectors @ vectors.T
    for i, language in enumerate(texts["lang"]):
        similarities[i, texts["lang"].eq(language).to_numpy()] = -np.inf
    nearest = similarities.argmax(axis=1)
    neighbours = pd.DataFrame({
        "text": texts["text"],
        "nearest_other_language": texts.iloc[nearest]["text"].to_numpy(),
        "own_label": texts["gold"],
        "neighbour_label": texts.iloc[nearest]["gold"].to_numpy(),
    })
    display(neighbours)
else:
    print("Embedding extension skipped; measured examples are shown above.")


## Reading

- [Licht and Lind (2023)](https://www.aup-online.com/content/journals/10.5117/CCR2023.2.2.LICH)
- [Maier et al. (2021)](https://christian-baden.huji.ac.il/publications/machine-translation-vs-multilingual-dictionaries-assessing-two)
- [Mimno et al. (2009)](https://aclanthology.org/D09-1092/)
- [Conneau et al. (2020)](https://aclanthology.org/2020.acl-main.747/)
- [Feng et al. (2022)](https://aclanthology.org/2022.acl-long.62/)
- [Wang et al. (2024)](https://arxiv.org/abs/2402.05672)
